# Palmer Penguins | Flipper Length and Body Mass

Gerardo Oburu Ndong · November 23, 2026  
CIS215 Capstone 12.6

## Initial data exploration

I’m using the simplified Palmer Penguins CSV. The [data notes](../data/README.md) explain where it came from and who was sampled.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

%matplotlib inline

# I am going to support a kernel started at the project root or in notebooks/.
project_root = Path("..") if Path("../data/README.md").is_file() else Path(".")
data_path = project_root / "data" / "raw" / "penguins.csv"
if not data_path.is_file():
    raise FileNotFoundError("Open the notebook from the project root or notebooks folder.")

penguins_raw = pd.read_csv(data_path)
penguins_raw.info()
display(penguins_raw.describe(include="all").round(2))
display(penguins_raw.isna().sum().rename("missing_rows").to_frame())
print(f"Exact duplicate rows: {penguins_raw.duplicated().sum()}")

The exploration output shows 344 rows and eight columns. Both bill columns, flipper length, and body mass each have two missing values. Recorded sex is missing in 11 rows. I found no exact duplicate rows. The simplified file has no bird ID, so that check only tells me whether entire rows match.

## Data cleaning and preparation

| Column | How I handled missing values |
|---|---|
| Species | No missing values. A row without a species could not go into a species comparison. |
| Island | No missing values. Island is used to check coverage. |
| Bill length | Two missing values. I left them blank because bill length does not enter my analysis. |
| Bill depth | Two missing values. I left them blank for the same reason. |
| Flipper length | Two missing values. I excluded those rows from the paired analysis because a missing flipper length cannot be plotted or correlated with mass. |
| Body mass | Two missing values, on the same rows as flipper length. I excluded those rows once and kept the original CSV intact. |
| Sex | Eleven missing values in the original file. I kept unknown sex in the pooled and species summaries when both measurements exist. I used known values for the sex comparison. |
| Year | No missing values. I kept the original year to describe the field seasons. |

I trimmed spaces in the group labels, standardized recorded sex to lowercase, and changed the display spelling of Adelie to Adélie. I checked for nonpositive dimensions and used kilograms on charts because the labels are easier to read.

In [ ]:
cleaned = penguins_raw.copy()
for column in ["species", "island", "sex"]:
    cleaned[column] = cleaned[column].str.strip().replace("", pd.NA)
cleaned["sex"] = cleaned["sex"].str.lower()
cleaned["species"] = cleaned["species"].replace({"Adelie": "Adélie"})

measurements = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
assert not cleaned[measurements].le(0).any().any(), "Check nonpositive measurements"
assert not cleaned[["species", "island", "year"]].isna().any().any()
assert not cleaned.duplicated().any(), "Review matching rows before proceeding"

paired = cleaned.dropna(subset=["flipper_length_mm", "body_mass_g"]).copy()
paired["body_mass_kg"] = paired["body_mass_g"] / 1000
print(f"Paired analysis: {len(paired)} of {len(cleaned)} rows")
print(f"Paired rows with unknown sex: {paired['sex'].isna().sum()}")

## Analysis question

Among the penguins in this dataset, how closely does flipper length go with body mass, and how does that relationship change when I look at each species on its own?

In [ ]:
# Pearson r measures the direction and strength of a linear relationship.
variables = ["flipper_length_mm", "body_mass_kg"]
display(paired[variables].describe().round(2))

overall_r = paired["flipper_length_mm"].corr(paired["body_mass_kg"])
species_groups = paired.groupby("species", sort=True)
species_summary = species_groups.agg(
    observations=("body_mass_kg", "size"),
    mean_flipper_mm=("flipper_length_mm", "mean"),
    mean_mass_kg=("body_mass_kg", "mean"),
)
species_summary["pearson_r"] = pd.Series({
    species: group["flipper_length_mm"].corr(group["body_mass_kg"])
    for species, group in species_groups
})
print(f"Pooled Pearson r: {overall_r:.3f}")
display(species_summary.round(3))

The pooled correlation is 0.87 across 342 measurements. The species results run from 0.47 for Adélie to 0.70 for Gentoo. Gentoo also had the longest average flippers (217.2 mm) and the highest average mass (5.08 kg). When I put all three species on one chart, that separation helps explain why the overall relationship looks stronger.

In [ ]:
figure_dir = project_root / "output" / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
colors = ["#27678e", "#9b5c18", "#28694f"]
markers = ["o", "^", "s"]

fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")
for (species, group), color, marker in zip(species_groups, colors, markers):
    ax.scatter(group["flipper_length_mm"], group["body_mass_kg"],
               label=species, color=color, marker=marker, alpha=0.73, s=32)
ax.set(title="Penguin flipper length and body mass",
       xlabel="Flipper length (mm)", ylabel="Body mass (kg)")
ax.legend(title="Species", frameon=False)
fig.savefig(figure_dir / "flipper_mass_scatter.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")
groups = [group["body_mass_kg"] for _, group in species_groups]
positions = range(1, len(groups) + 1)
ax.boxplot(groups, positions=positions, widths=0.48,
           medianprops={"color": "#202020", "linewidth": 2})
ax.set_xticks(list(positions), [f"{name} (n={len(group)})" for name, group in species_groups])
ax.set(title="Body mass distribution by species", xlabel="Species", ylabel="Body mass (kg)")
fig.savefig(figure_dir / "body_mass_by_species.png", dpi=150)
plt.show()

In [ ]:
correlations = pd.concat([pd.Series({"All penguins": overall_r}), species_summary["pearson_r"]])
fig, ax = plt.subplots(figsize=(8, 4.8), layout="constrained")
bars = ax.barh(correlations.index, correlations, color=["#343434"] + colors, edgecolor="#202020")
bars[0].set_hatch("//")
ax.invert_yaxis()
ax.set(title="Flipper length and body mass correlation by group",
       xlabel="Pearson correlation (r)", ylabel="Group", xlim=(-1, 1))
ax.axvline(0, color="#333333", linewidth=0.8)
ax.bar_label(bars, fmt="%.2f", padding=4)
fig.savefig(figure_dir / "correlation_by_group.png", dpi=150)
plt.show()

## Sample coverage and recorded sex

In [ ]:
display(pd.crosstab(cleaned["species"], cleaned["island"]))
display(pd.crosstab(cleaned["species"], cleaned["year"]))
display(pd.crosstab(cleaned["species"], cleaned["sex"].fillna("Unknown")))

known_sex = paired.dropna(subset=["sex"])
sex_summary = pd.DataFrame([
    {"species": species, "recorded_sex": sex, "observations": len(group),
     "mean_mass_kg": group["body_mass_kg"].mean(),
     "pearson_r": group["flipper_length_mm"].corr(group["body_mass_kg"])}
    for (species, sex), group in known_sex.groupby(["species", "sex"])
]).set_index(["species", "recorded_sex"])
display(sex_summary.round(3))

## Limitations and Bias Considerations

- **Collection.** The island table shows all 68 Chinstrap observations came from Dream and all 124 Gentoo observations came from Biscoe. Only Adélie appears on all three islands. The study covers adult foragers, so chicks and birds in other colonies are missing. The sampling design makes it hard to separate species patterns from site conditions.
- **Measurement.** The CSV gives dimensions in millimeters and mass in grams, but it has no observer or instrument fields. I cannot check whether measurement differences across field crews affected the relationship. Two rows lack the paired measurements. Nine usable pairs have unknown recorded sex, so the sex summaries describe 333 known-sex observations.
- **Historical coverage.** The data come from 2007–2009. There are only three field seasons here, so I cannot use them to describe today’s penguins or estimate a long-term trend.
- **Recorded sex.** In the subgroup table, Chinstrap correlation was about 0.24 for females and 0.66 for males, with 34 observations in each group. The male mean mass was higher within all three species. These sample comparisons can be influenced by group size, missing data, and collection conditions. I did not test whether the differences would hold in the broader population.